# Wskaźnik wykluczenia heksagonu — MLP (SiLU / GELU) — Google Colab (hackathon)

Sieć neuronowa (PyTorch MLP) przewidująca `etykieta_target` (0–1) dla heksagonów H3.
Dane pobierane są z **prywatnego repo GitHub przez token** (nic nie trzeba wgrywać ręcznie).

**Plik z repo:** `dane/wskaznik_wykluczenia/h3_merged_cleanest.csv` (pakiet podstawowy + rozszerzony, 39 kolumn).
**Wymagania:** Colab (CPU wystarczy), token GitHub z prawem `repo` (klasyczny) lub `contents:read` (fine-grained).
**Output:** `lista_heksagonow_ranking_cleanest.csv` + `heatmapa_wykluczenia.html` + `heatmapa_wykluczenia.geojson`.

Względem wersji bazowej: więcej cech (kursy, odległości do metra/kolei, Veturilo, czas do centrum, luka podaży, seniorzy) + **mały grid** (screening sieć × etykieta + re-trening zwycięzcy).


In [ ]:
# ===== KONFIGURACJA =====
REPO = "Hackathon-42Warsaw-The-Bills/3_TheBills_OptymalnaWarszawa"  # wlasciciel/nazwa
BRANCH = "wskaznik_wykluczenia_klastra"  # galaz z danymi (main NIE ma tych plikow)
FALLBACK_BRANCHES = ["main"]  # probowane po kolei gdy BRANCH zwroci 404
FILES = [
    "dane/wskaznik_wykluczenia/h3_merged_cleanest.csv",
    "dane/klastry/kontury_400m/warszawa_h3_400m_pelna.geojson",  # geometria do heatmapy
]  # pliki pobierane z repo
LOCAL_DIR = "/content/dane_repo"  # gdzie zapisac pobrane pliki

# Parametry etykiety (musza zgadzac sie z compute_target.py)
ALPHA, W_TRAM, W_BUS, W_METRO, W_KOLEJ = 0.3, 1.5, 1.0, 3.0, 2.5

# Parametry sieci
HIDDEN = [32, 16]
HIDDEN_ACT = "silu"   # 'silu' | 'gelu' | 'relu'
OUTPUT_ACT = "silu"   # 'silu' | 'gelu' | 'none'  (predykcje klampowane do [0,1])
DROPOUT = 0.3
LR, WEIGHT_DECAY, BATCH, EPOCHS, PATIENCE, SEED = 1e-3, 1e-4, 64, 300, 30, 42

In [ ]:
# ===== TOKEN GITHUB (nie wpisuj na stale do notebooka) =====
import getpass, os
TOKEN = None
try:
    from google.colab import userdata  # Colab: zakladka Secrets (klucz: GITHUB_TOKEN_HACKATHON)
    TOKEN = userdata.get('GITHUB_TOKEN_HACKATHON')
    if TOKEN: print('token: Colab Secrets')
except Exception:
    pass
if not TOKEN:
    TOKEN = os.environ.get("GITHUB_TOKEN_HACKATHON")
    if TOKEN: print('token: zmienna srodowiskowa')
if not TOKEN:
    TOKEN = getpass.getpass("GitHub token (niewidoczny): ")
assert TOKEN and TOKEN.strip(), "Podaj token GitHub"
os.environ["GITHUB_TOKEN_HACKATHON"] = TOKEN  # zeby nie pytac ponownie po resecie UI
# szybki test uprawnien tokenu (contents:read do repo)
import urllib.request as _u
try:
    _r = _u.Request(f"https://api.github.com/repos/{REPO}/contents/", headers={"Authorization": f"Bearer {TOKEN}"})
    print('uprawnienia tokenu: OK, status', _u.urlopen(_r).status)
except Exception as e:
    print('UWAGA: test tokenu nie przeszedl:', e)

In [ ]:
# ===== POBRANIE PLIKOW Z REPO PRZEZ TOKEN =====
import os, urllib.request
os.makedirs(LOCAL_DIR, exist_ok=True)

def fetch_repo_file(repo_path, branch=BRANCH):
    last_err = None
    for br in [branch] + [b for b in FALLBACK_BRANCHES if b != branch]:
        url = f"https://raw.githubusercontent.com/{REPO}/{br}/{repo_path}"
        req = urllib.request.Request(url, headers={
            "Authorization": f"Bearer {TOKEN}",
            "Accept": "application/vnd.github.raw",
        })
        local = os.path.join(LOCAL_DIR, os.path.basename(repo_path))
        try:
            with urllib.request.urlopen(req) as r, open(local, "wb") as f:
                f.write(r.read())
            print(f"OK: {repo_path}@{br} -> {local} ({os.path.getsize(local)} B)")
            return local
        except Exception as e:
            last_err = f"[{br}] {e}"
    raise RuntimeError(f"Nie pobrano {repo_path}: {last_err}")

LOCAL = {p: fetch_repo_file(p) for p in FILES}
CSV_PATH = LOCAL["dane/wskaznik_wykluczenia/h3_merged_cleanest.csv"]
GEO_PATH = LOCAL["dane/klastry/kontury_400m/warszawa_h3_400m_pelna.geojson"]  # do heatmapy

In [ ]:
# ===== ZALEZNOSCI / IMPORTY =====
# Colab ma zwykle torch/sklearn/pandas; dociągnij tylko brakujące
import importlib, subprocess, sys
for pkg, mod in [("torch", "torch"), ("scikit-learn", "sklearn"), ("pandas", "pandas"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("folium", "folium")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])

import math, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr
%matplotlib inline
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# ===== WCZYTANIE DANYCH + OBSLUGA BRAKOW =====
df = pd.read_csv(CSV_PATH)
print(df.shape, list(df.columns))
assert "etykieta_target" in df.columns, "Brak etykiety w CSV"

# metro_plan_rok: puste = brak planowanego metra -> flaga 0/1 (rok odrzucamy)
df["ma_metro_plan"] = (~df["metro_plan_rok"].isna()).astype(float)
# ma_veturilo: tekst True/False -> 0/1
df["ma_veturilo"] = df["ma_veturilo"].astype(str).str.lower().map({"true": 1.0, "false": 0.0, "1": 1.0, "0": 0.0}).fillna(0.0)
# czas_do_centrum: 'inf' (brak dojazdu) -> NaN -> mediana
df["czas_do_centrum_min_2025"] = pd.to_numeric(df["czas_do_centrum_min_2025"], errors="coerce").replace([np.inf, -np.inf], np.nan)
df["czas_do_centrum_min_2025"] = df["czas_do_centrum_min_2025"].fillna(df["czas_do_centrum_min_2025"].median())
# v_autobus: 0 = brak autobusow (rozroznia kolumna kursow); puste -> mediana
df["v_autobus_szczyt_kmh"] = pd.to_numeric(df["v_autobus_szczyt_kmh"], errors="coerce")
df["v_autobus_szczyt_kmh"] = df["v_autobus_szczyt_kmh"].fillna(df["v_autobus_szczyt_kmh"].median())
# udzial_65_: pojedyncze braki -> mediana
df["udzial_65_"] = pd.to_numeric(df["udzial_65_"], errors="coerce").fillna(df["udzial_65_"].median())
# reszta numeryczna -> float, braki -> 0
NUM_EXTRA = ["kursy_dzien_autobus", "kursy_dzien_tramwaj", "kursy_dzien_metro", "kursy_dzien_kolej",
             "kursy_szczyt_bus_tram", "linie_ztm", "odl_centrum_km",
             "dist_metro_m", "dist_kolej_m", "odl_do_veturilo_m",
             "roznica_dzis", "pop_2031", "wsiadajacy_h_2025",
             "infra_handel", "infra_kult", "infra_kultura", "infra_park",
             "infra_szkola", "infra_szpital", "infra_targowisko", "infra_urzad"]
df[NUM_EXTRA] = df[NUM_EXTRA].apply(pd.to_numeric, errors="coerce").fillna(0.0)
print("braki po obsludze:", int(df.isna().sum().sum()))
print(df["etykieta_target"].describe().to_string())

In [ ]:
# ===== CECHY: log1p + one-hot dzielnicy + standaryzacja =====
USE_DZIELNICA = True
feat = df.copy()
feat["log_gest"] = np.log1p(feat["gestosc_os_km2"].clip(lower=0))
feat["log_prac"] = np.log1p(feat["pracownicy_szac"].clip(lower=0))
feat["ztm_w"] = W_TRAM * feat["przystanki_tramwajowe"] + W_BUS * feat["przystanki_autobusowe"] + W_METRO * (feat["czy_metro"] >= 0.5).astype(float) + W_KOLEJ * (feat["czy_kolej"] >= 0.5).astype(float)
feat["log_ztm"] = np.log1p(feat["ztm_w"].clip(lower=0))
feat["log_kursy"] = np.log1p((feat[["kursy_dzien_autobus", "kursy_dzien_tramwaj", "kursy_dzien_metro", "kursy_dzien_kolej"]].clip(lower=0)).sum(axis=1))
feat["log_kursy_szczyt"] = np.log1p(feat["kursy_szczyt_bus_tram"].clip(lower=0))
feat["log_linie"] = np.log1p(feat["linie_ztm"].clip(lower=0))
feat["log_d_metro"] = np.log1p(feat["dist_metro_m"].clip(lower=0))
feat["log_d_kolej"] = np.log1p(feat["dist_kolej_m"].clip(lower=0))
feat["log_vet"] = np.log1p(feat["odl_do_veturilo_m"].clip(lower=0))
feat["log_wsiad"] = np.log1p(feat["wsiadajacy_h_2025"].clip(lower=0))
feat["log_pop31"] = np.log1p(feat["pop_2031"].clip(lower=0))
feat["infra_sum"] = feat[["infra_handel", "infra_kult", "infra_kultura", "infra_park", "infra_szkola", "infra_szpital", "infra_targowisko", "infra_urzad"]].sum(axis=1)
base = ["log_gest", "log_prac", "log_ztm", "log_kursy", "log_kursy_szczyt", "log_linie",
        "log_d_metro", "log_d_kolej", "ma_metro_plan", "v_autobus_szczyt_kmh",
        "udzial_65_", "odl_centrum_km", "log_vet", "ma_veturilo",
        "czas_do_centrum_min_2025", "roznica_dzis", "log_pop31", "log_wsiad",
        "infra_sum", "wspolczynnik przyrostu", "lon", "lat"]
X_num = feat[base].to_numpy(float)
if USE_DZIELNICA:
    dummies = pd.get_dummies(feat["dzielnica"].fillna("UNK"), prefix="dz").to_numpy(float)
    X = np.hstack([X_num, dummies])
    feat_names = base + list(pd.get_dummies(feat["dzielnica"].fillna("UNK"), prefix="dz").columns)
else:
    X, feat_names = X_num, base
y = feat["etykieta_target"].to_numpy(float)
print("cechy:", X.shape, "|", len(feat_names))

In [ ]:
# ===== PODZIAL PRZESTRZENNY (po dzielnicach, nie losowo) =====
groups = feat["dzielnica"].fillna("UNK").to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr, te = next(gss.split(X, y, groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=SEED)  # ~15% calosci na walidacje
tr2, va = next(gss2.split(X[tr], y[tr], groups[tr]))
tr_idx, va_idx, te_idx = tr[tr2], tr[va], te
sc = StandardScaler().fit(X[tr_idx])
Xs = sc.transform(X[tr_idx]), sc.transform(X[va_idx]), sc.transform(X[te_idx])
print(f"train/val/test: {len(tr_idx)}/{len(va_idx)}/{len(te_idx)}")
print("dzielnice testowe:", sorted(set(groups[te_idx])))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# ===== MODEL: MLP z SiLU/GELU =====
ACTS = {"silu": nn.SiLU(), "gelu": nn.GELU(), "relu": nn.ReLU()}

class ExclusionMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, hidden_act=HIDDEN_ACT, output_act=OUTPUT_ACT, dropout=DROPOUT):
        super().__init__()
        h, layers, prev = list(hidden), [], in_dim
        for w in h:
            layers += [nn.Linear(prev, w), ACTS[hidden_act], nn.Dropout(dropout)]
            prev = w
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
        self.out = {"silu": nn.SiLU(), "gelu": nn.GELU(), "none": nn.Identity()}[output_act]
    def forward(self, x):
        return self.out(self.net(x)).squeeze(-1)

model = ExclusionMLP(X.shape[1]).to(device)
print(model)
print("parametry:", sum(p.numel() for p in model.parameters()))

In [ ]:
# ===== TRENING: Adam + MSE + early stopping =====
def to_t(a): return torch.tensor(a, dtype=torch.float32, device=device)
Xtr, Xva, Xte = map(to_t, Xs)
ytr, yva, yte = map(to_t, (y[tr_idx], y[va_idx], y[te_idx]))
tr_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xtr, ytr), batch_size=BATCH, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=15)
lossf = nn.MSELoss()
hist, best, bad, best_state = {"tr": [], "va": []}, float("inf"), 0, None
for ep in range(1, EPOCHS + 1):
    model.train()
    tl = 0.0
    for xb, yb in tr_loader:
        opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
        tl += loss.item() * len(xb)
    tl /= len(Xtr)
    model.eval()
    with torch.no_grad():
        vl = lossf(model(Xva), yva).item()
    sched.step(vl); hist["tr"].append(tl); hist["va"].append(vl)
    if vl < best - 1e-5: best, bad, best_state = vl, 0, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else: bad += 1
    if ep % 25 == 0 or bad == 0: print(f"ep {ep:3d} train={tl:.4f} val={vl:.4f} best={best:.4f}")
    if bad >= PATIENCE: print(f"early stop @ {ep}"); break
model.load_state_dict(best_state)
plt.plot(hist["tr"], label="train MSE"); plt.plot(hist["va"], label="val MSE")
plt.legend(); plt.xlabel("epoka"); plt.grid(True); plt.show()

In [ ]:
# ===== EWALUACJA + RANKING =====
model.eval()
with torch.no_grad():
    pred_all = model(to_t(sc.transform(X))).cpu().numpy()
pred_all = np.clip(pred_all, 0, 1)  # SiLU/GELU na wyjsciu -> dociecie do zakresu etykiety
te_pred = pred_all[te_idx]
print(f"TEST MAE={mean_absolute_error(y[te_idx], te_pred):.4f} RMSE={math.sqrt(mean_squared_error(y[te_idx], te_pred)):.4f} R2={r2_score(y[te_idx], te_pred):.4f} Spearman={spearmanr(y[te_idx], te_pred).statistic:.4f}")
for k in (20, 50):
    t, p = set(np.argsort(-y[te_idx])[:k]), set(np.argsort(-te_pred)[:k])
    print(f"Precision@{k} (test) = {len(t & p) / k:.2f}")
plt.scatter(y[te_idx], te_pred, s=8); plt.plot([0, 1], [0, 1], "r--")
plt.xlabel("etykieta"); plt.ylabel("predykcja"); plt.grid(True); plt.show()

In [ ]:
# ===== OUTPUT: lista heksagonow =====
out = df[["h3", "dzielnica", "lon", "lat", "etykieta_target"]].copy()
out["pred_wykluczenie"] = np.round(pred_all, 6)
out = out.sort_values("pred_wykluczenie", ascending=False).reset_index(drop=True)
out["rank"] = out.index + 1
out.to_csv("/content/lista_heksagonow_ranking_cleanest.csv", index=False)
torch.save(model.state_dict(), "/content/model_cleanest.pt")
print(out.head(20).to_string(index=False))
print("zapisano: /content/lista_heksagonow_ranking_cleanest.csv, /content/model_cleanest.pt")

## Mini-grid: sieć × etykieta (screening)

Grid po sieci `HIDDEN × ACT × LR` (2×2×2 = 8) **razy** grid etykiety
`ALPHA × W_TRAM × W_BUS × W_METRO × W_KOLEJ` (po 2 wartości).
Tryb `coord` (domyślny): baza + 5 wariantów po 1 zmianie = 6 kombinacji etykiety × 8 sieci = **48 treningów**.
Tryb `full`: pełny kartezjan 32 × 8 = **256 treningów** (długo na CPU — tylko na noc).
Selekcja po **val Spearman** (ranking heksagonów).


In [ ]:
# ===== MINI-GRID: siec x etykieta =====
import itertools, math, time
GRID_NET = {"HIDDEN": [[32, 16], [64, 32]], "ACT": ["silu", "gelu"], "LR": [1e-3, 3e-4]}
GRID_LABEL = {"ALPHA": [0.2, 0.3], "W_TRAM": [1.0, 1.5], "W_BUS": [0.5, 1.0],
              "W_METRO": [2.0, 3.0], "W_KOLEJ": [2.0, 2.5]}
BASE_LABEL = {"ALPHA": ALPHA, "W_TRAM": W_TRAM, "W_BUS": W_BUS,
              "W_METRO": W_METRO, "W_KOLEJ": W_KOLEJ}
GRID_WD, GRID_SEED = 1e-4, 42
GRID_MODE = "coord"  # 'coord' = baza + 5 wariantow po 1 zmianie (6x8=48) | 'full' = 32x8=256

KEYS_L = ["ALPHA", "W_TRAM", "W_BUS", "W_METRO", "W_KOLEJ"]
if GRID_MODE == "full":
    label_combos = [dict(zip(KEYS_L, v)) for v in itertools.product(*(GRID_LABEL[k] for k in KEYS_L))]
else:
    label_combos = [dict(BASE_LABEL)]
    for k in KEYS_L:
        alt = [v for v in GRID_LABEL[k] if v != BASE_LABEL[k]]
        c = dict(BASE_LABEL); c[k] = alt[0] if alt else BASE_LABEL[k]
        if c not in label_combos:
            label_combos.append(c)
net_combos = list(itertools.product(GRID_NET["HIDDEN"], GRID_NET["ACT"], GRID_NET["LR"]))
print(f"konfigi etykiety: {len(label_combos)} x sieci: {len(net_combos)} = {len(label_combos)*len(net_combos)} treningow")

def _pct(sv, q):
    n = len(sv)
    if n == 1:
        return sv[0]
    r = q / 100.0 * (n - 1)
    lo, hi = int(math.floor(r)), int(math.ceil(r))
    return sv[lo] if lo == hi else sv[lo] * (1 - (r - lo)) + sv[hi] * (r - lo)

def compute_label_vals(frame, alpha, w_tram, w_bus, w_metro, w_kolej):
    # wzor z compute_target.py: M*(P+J) - Z, norm. p1/p99
    gest = frame["gestosc_os_km2"].clip(lower=0).to_numpy(float)
    prac = frame["pracownicy_szac"].clip(lower=0).to_numpy(float)
    tram = frame["przystanki_tramwajowe"].clip(lower=0).to_numpy(float)
    bus = frame["przystanki_autobusowe"].clip(lower=0).to_numpy(float)
    metro = (frame["czy_metro"].to_numpy(float) >= 0.5).astype(float)
    kolej = (frame["czy_kolej"].to_numpy(float) >= 0.5).astype(float)
    d = np.clip(pd.to_numeric(frame["wspolczynnik przyrostu"], errors="coerce").fillna(0).to_numpy(float), 0, 1)
    lp, lj = np.log1p(gest), np.log1p(prac)
    lz = np.log1p(w_tram * tram + w_bus * bus + w_metro * metro + w_kolej * kolej)
    def mm(v):
        lo, hi = v.min(), v.max()
        return np.zeros_like(v) if hi - lo <= 0 else (v - lo) / (hi - lo)
    pn, jn, zn = mm(lp), mm(lj), mm(lz)
    raw = (alpha + alpha * d) * (pn + jn) - zn
    s = sorted(raw.tolist())
    p1, p99 = _pct(s, 1), _pct(s, 99)
    return np.clip((raw - p1) / (p99 - p1), 0, 1) if p99 - p1 > 0 else np.zeros_like(raw)

DUM_COLS = list(feat_names[len(base):])  # kolejnosc one-hot z komorki cech
def build_grid_Xy(frame, feat_ref, base_cols, dum_cols, lab):
    f = feat_ref.copy()
    f["ztm_w"] = lab["W_TRAM"] * f["przystanki_tramwajowe"] + lab["W_BUS"] * f["przystanki_autobusowe"] \
        + lab["W_METRO"] * (f["czy_metro"] >= 0.5).astype(float) + lab["W_KOLEJ"] * (f["czy_kolej"] >= 0.5).astype(float)
    f["log_ztm"] = np.log1p(f["ztm_w"].clip(lower=0))
    Xn = f[base_cols].to_numpy(float)
    dd = pd.get_dummies(f["dzielnica"].fillna("UNK"), prefix="dz").reindex(columns=dum_cols, fill_value=0).to_numpy(float)
    return np.hstack([Xn, dd]), compute_label_vals(frame, lab["ALPHA"], lab["W_TRAM"], lab["W_BUS"], lab["W_METRO"], lab["W_KOLEJ"])

def grid_train(Xtr, ytr, Xva, yva, hidden, act, lr, wd, seed, epochs=EPOCHS, patience=PATIENCE):
    torch.manual_seed(seed); np.random.seed(seed)
    m = ExclusionMLP(Xtr.shape[1], hidden=hidden, hidden_act=act, output_act=act,
                     dropout=DROPOUT).to(device)
    dl = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                                       torch.tensor(ytr, dtype=torch.float32)),
        batch_size=BATCH, shuffle=True)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=15)
    lf, best, bad, bs = nn.MSELoss(), float("inf"), 0, None
    Xt = torch.tensor(Xtr, dtype=torch.float32, device=device)
    Xv = torch.tensor(Xva, dtype=torch.float32, device=device)
    Yv = torch.tensor(yva, dtype=torch.float32, device=device)
    for _ in range(epochs):
        m.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); l = lf(m(xb), yb); l.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            vl = lf(m(Xv), Yv).item()
        sched.step(vl)
        if vl < best - 1e-5: best, bad, bs = vl, 0, {k: v.cpu().clone() for k, v in m.state_dict().items()}
        else: bad += 1
        if bad >= patience: break
    m.load_state_dict(bs)
    m.eval()
    with torch.no_grad():
        pv = m(Xv).cpu().numpy()
    pv = np.clip(pv, 0, 1)
    return math.sqrt(mean_squared_error(yva, pv)), spearmanr(yva, pv).statistic, m

rows_g, t0 = [], time.time()
for lc in label_combos:
    Xg, yg = build_grid_Xy(df, feat, base, DUM_COLS, lc)
    sc_g = StandardScaler().fit(Xg[tr_idx])
    Xtr0, Xva0 = sc_g.transform(Xg[tr_idx]), sc_g.transform(Xg[va_idx])
    ytr0, yva0 = yg[tr_idx], yg[va_idx]
    for hidden, act, lr in net_combos:
        rmse, sp, _ = grid_train(Xtr0, ytr0, Xva0, yva0, hidden, act, lr, GRID_WD, GRID_SEED)
        rows_g.append({"alpha": lc["ALPHA"], "w_tram": lc["W_TRAM"], "w_bus": lc["W_BUS"],
                       "w_metro": lc["W_METRO"], "w_kolej": lc["W_KOLEJ"],
                       "hidden": str(hidden), "act": act, "lr": lr,
                       "val_rmse": round(rmse, 5), "val_spearman": round(float(sp), 5)})
        print(f"a={lc['ALPHA']} tram={lc['W_TRAM']} bus={lc['W_BUS']} metro={lc['W_METRO']} kolej={lc['W_KOLEJ']} | "
              f"{hidden} {act} lr={lr}: rmse={rmse:.5f} spearman={sp:.5f} [{(time.time()-t0)/60:.1f} min]")
res = pd.DataFrame(rows_g).sort_values("val_spearman", ascending=False).reset_index(drop=True)
res.to_csv("/content/grid_screening_cleanest.csv", index=False)
print(res.to_string(index=False))
WIN = res.iloc[0].to_dict()
w_alpha, w_tram, w_bus, w_metro, w_kolej = WIN["alpha"], WIN["w_tram"], WIN["w_bus"], WIN["w_metro"], WIN["w_kolej"]
print("ZWYCIEZCA:", WIN)


In [ ]:
# ===== FINAL: re-trening zwyciezcy na train+val, test raz, ranking =====
w_hidden = [int(x) for x in str(WIN["hidden"]).strip("[]").split(",")]
w_alpha, w_tram, w_bus, w_metro, w_kolej = WIN["alpha"], WIN["w_tram"], WIN["w_bus"], WIN["w_metro"], WIN["w_kolej"]
Xf, yf = build_grid_Xy(df, feat, base, DUM_COLS,
                       {"ALPHA": w_alpha, "W_TRAM": w_tram, "W_BUS": w_bus, "W_METRO": w_metro, "W_KOLEJ": w_kolej})
sc_f = StandardScaler().fit(Xf[np.concatenate([tr_idx, va_idx])])
Xtrv = sc_f.transform(Xf[np.concatenate([tr_idx, va_idx])])
ytrv = yf[np.concatenate([tr_idx, va_idx])]
Xte_f = sc_f.transform(Xf[te_idx])
torch.manual_seed(GRID_SEED); np.random.seed(GRID_SEED)
final = ExclusionMLP(Xf.shape[1], hidden=w_hidden, hidden_act=WIN["act"],
                     output_act=WIN["act"], dropout=DROPOUT).to(device)
dlf = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(
    torch.tensor(Xtrv, dtype=torch.float32), torch.tensor(ytrv, dtype=torch.float32)),
    batch_size=BATCH, shuffle=True)
optf = torch.optim.Adam(final.parameters(), lr=WIN["lr"], weight_decay=GRID_WD)
lf = nn.MSELoss()
final.train()
for _ in range(EPOCHS):
    for xb, yb in dlf:
        xb, yb = xb.to(device), yb.to(device)
        optf.zero_grad(); l = lf(final(xb), yb); l.backward(); optf.step()
final.eval()
with torch.no_grad():
    pall = np.clip(final(torch.tensor(sc_f.transform(Xf), dtype=torch.float32, device=device)).cpu().numpy(), 0, 1)
pte = pall[te_idx]
print(f"ETYKIETA FINAL: alpha={w_alpha} tram={w_tram} bus={w_bus} metro={w_metro} kolej={w_kolej}")
print(f"TEST: MAE={mean_absolute_error(yf[te_idx], pte):.4f} "
      f"RMSE={math.sqrt(mean_squared_error(yf[te_idx], pte)):.4f} R2={r2_score(yf[te_idx], pte):.4f} "
      f"Spearman={spearmanr(yf[te_idx], pte).statistic:.4f}")
out = df[["h3", "dzielnica", "lon", "lat"]].copy()
out["etykieta_final"] = np.round(yf, 6)
out["pred_wykluczenie"] = np.round(pall, 6)
out = out.sort_values("pred_wykluczenie", ascending=False).reset_index(drop=True)
out["rank"] = out.index + 1
out.to_csv("/content/lista_heksagonow_ranking_best.csv", index=False)
TOP_K = 30  # liczba heksagonow o najwyzszym wskazniku do osobnego pliku
out.head(TOP_K).to_csv("/content/lista_heksagonow_top30.csv", index=False)
torch.save({"state": final.state_dict(), "winner": WIN, "scaler_mean": sc_f.mean_, "scaler_scale": sc_f.scale_},
           "/content/model_best.pt")
print(out.head(20).to_string(index=False))
print("zapisano: /content/lista_heksagonow_ranking_best.csv, /content/lista_heksagonow_top30.csv, /content/model_best.pt")


## Heatmapa heksagonów (folium) + GeoJSON do pobrania

Wymaga uruchomienia komórki finałowej (`out` z rankingiem). Kolor = `pred_wykluczenie`, czarny obrys = top-30.
Zapisy: `/content/heatmapa_wykluczenia.html` (podgląd) + `/content/heatmapa_wykluczenia.geojson` (do pobrania z panelu Files — properties: `pred`, `rank`, `top30`, `status`).

In [ ]:
# ===== HEATMAPA: join geometrii z rankingiem + GEOJSON =====
import folium, json
from branca.colormap import LinearColormap
try:
    GEO_PATH  # z komorki pobierania; jesli brak -> odtworz sciezke
except NameError:
    GEO_PATH = os.path.join(LOCAL_DIR, "warszawa_h3_400m_pelna.geojson")
try:
    TOP_K
except NameError:
    TOP_K = 30
rank_df = out[["h3", "dzielnica", "pred_wykluczenie", "rank"]].copy()
with open(GEO_PATH, encoding="utf-8") as f:
    geo = json.load(f)
pred = dict(zip(rank_df["h3"], rank_df["pred_wykluczenie"]))
rnk = dict(zip(rank_df["h3"], rank_df["rank"]))
matched, missing = 0, []
for feat in geo["features"]:
    h = feat["properties"].get("h3")
    if h in pred:
        feat["properties"]["pred"] = round(float(pred[h]), 4)
        feat["properties"]["rank"] = int(rnk[h])
        feat["properties"]["status"] = "analiza"
        matched += 1
    else:
        feat["properties"]["pred"] = -1.0
        feat["properties"]["rank"] = -1
        feat["properties"]["status"] = "poza analiza (clean)"
        missing.append(h)
print(f"dopasowano {matched}/{len(geo['features'])} geometrii" + (f", brak: {missing[:5]}" if missing else ""))
cmap = LinearColormap(["#ffffcc", "#fed976", "#feb24c", "#fd8d3c", "#e31a1c", "#800026"],
                      vmin=0, vmax=1, caption="pred_wykluczenie")
top30 = set(rank_df[rank_df["rank"] <= TOP_K]["h3"])
m = folium.Map(location=[float(out["lat"].mean()), float(out["lon"].mean())],
               zoom_start=11, tiles="CartoDB positron")
folium.GeoJson(
    geo, name="wskaznik wykluczenia",
    style_function=lambda f: ({"fillColor": "#cccccc", "color": "#888888", "weight": 0.3, "fillOpacity": 0.4}
                              if f["properties"].get("status") != "analiza" else
                              {"fillColor": cmap(f["properties"]["pred"]),
                              "color": "black", "weight": 0.3, "fillOpacity": 0.7}),
    tooltip=folium.GeoJsonTooltip(fields=["h3", "dzielnica", "pred", "rank", "status"],
                                  aliases=["h3", "dzielnica", "pred", "rank", "status"]),
).add_to(m)
folium.GeoJson(
    {"type": "FeatureCollection",
     "features": [f for f in geo["features"] if f["properties"].get("h3") in top30]},
    name=f"top-{TOP_K}", style_function=lambda f: {"color": "black", "weight": 2.5, "fillOpacity": 0},
).add_to(m)
cmap.add_to(m); folium.LayerControl().add_to(m)
try:
    _banner = f"(alpha={w_alpha} tram={w_tram} metro={w_metro} kolej={w_kolej})"
except NameError:
    _banner = f"(alpha={ALPHA} tram={W_TRAM} metro={W_METRO} kolej={W_KOLEJ})"
m.get_root().html.add_child(folium.Element(
    f"<div style='position:fixed;top:10px;left:50px;z-index:999;background:white;padding:6px 10px;"
    f"border:1px solid grey;font-size:13px'>Wskaznik wykluczenia — top-{TOP_K} obrysowane "
    f"{_banner}</div>"))
m.save("/content/heatmapa_wykluczenia.html")
print("zapisano: /content/heatmapa_wykluczenia.html")
for feat in geo["features"]:  # flaga top-K + zapis jako GeoJSON do pobrania
    feat["properties"]["top30"] = 1 if feat["properties"].get("h3") in top30 else 0
with open("/content/heatmapa_wykluczenia.geojson", "w", encoding="utf-8") as f:
    json.dump(geo, f, ensure_ascii=False)
print("zapisano: /content/heatmapa_wykluczenia.geojson")
m
